<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_10_1_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
**Modul 10: Zeitreihen in PyTorch**

* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Modul 10 Material

* **Teil 10.1: Zeitreihendatenkodierung für Deep Learning, PyTorch** [[Video]](https://www.youtube.com/watch?v=CZi5Avp6p1s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=CZi5Avp6p1s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 10.2: LSTM-basierte Zeitreihen mit PyTorch [[Video]](https://www.youtube.com/watch?v=hIQLy5zCgH4&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=hIQLy5zCgH4&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 10.3: Transformer-basierte Zeitreihen mit PyTorch [[Video]](https://www.youtube.com/watch?v=NGzQpphf_Vc&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=NGzQpphf_Vc&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 10.4: Saisonalität und Trend [[Video]](https://www.youtube.com/watch?v=HOkxoLaUF9s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=HOkxoLaUF9s&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 10.5: Vorhersagen mit Meta Prophet [[Video]](https://www.youtube.com/watch?v=MzjMVsz0GyA&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=MzjMVsz0GyA&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)

# Google CoLab-Anweisungen

Der folgende Code überprüft, ob Google CoLab installiert ist, und richtet die richtigen Hardwareeinstellungen für PyTorch ein.


In [1]:
try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")

Note: not using Google CoLab


# Teil 10.1: Zeitreihendatenkodierung

Es gibt viele verschiedene Methoden, um Daten über einen bestimmten Zeitraum in ein neuronales Netzwerk zu kodieren. In diesem Kapitel werden wir uns mit Zeitreihenkodierung und rekurrenten Netzwerken befassen, zwei Themen, die logischerweise miteinander verknüpft werden können, da beide Methoden zum Umgang mit Daten sind, die sich über einen bestimmten Zeitraum erstrecken. Bei der Zeitreihenkodierung geht es darum, Ereignisse, die über einen bestimmten Zeitraum auftreten, in einem neuronalen Netzwerk darzustellen. Diese Kodierung ist notwendig, da ein Feedforward-Neuronales Netzwerk für einen bestimmten Eingabevektor immer denselben Ausgabevektor erzeugt. Rekurrente neuronale Netzwerke erfordern keine Kodierung von Zeitreihendaten, da sie Daten, die über einen bestimmten Zeitraum auftreten, automatisch verarbeiten können.

Die Temperaturschwankungen während der Woche sind ein Beispiel für Zeitreihendaten. Wenn wir beispielsweise wissen, dass die Temperatur heute 25 Grad Fahrenheit und morgen 27 Grad beträgt, bieten die rekurrierenden neuronalen Netze und die Zeitreihenkodierung eine weitere Möglichkeit, die korrekte Temperatur für die Woche vorherzusagen. Umgekehrt wird ein traditionelles Feedforward-neuronales Netz für eine gegebene Eingabe immer mit derselben Ausgabe antworten. Wenn wir ein Feedforward-neuronales Netz trainieren, um die Temperatur von morgen vorherzusagen, sollte es für 25 einen Wert von 27 zurückgeben. Es wird immer 27 ausgeben, wenn 25 seine Vorhersagen beeinträchtigen könnte. Sicherlich wird die Temperatur von 27 nicht immer auf 25 folgen. Es wäre besser, wenn das neuronale Netz die Temperaturen der Tage vor der Vorhersage berücksichtigen würde. Vielleicht könnte uns die Temperatur der letzten Woche ermöglichen, die Temperatur von morgen vorherzusagen. Daher stellen rekurrierende neuronale Netze und die Zeitreihenkodierung zwei verschiedene Ansätze dar, um Daten im Zeitverlauf für ein neuronales Netz darzustellen.

Zuvor haben wir neuronale Netzwerke mit Eingabe ($x$) und erwarteter Ausgabe ($y$) trainiert. $X$ war eine Matrix, die Zeilen waren Trainingsbeispiele und die Spalten waren vorherzusagende Werte. Der $x$-Wert enthält jetzt Datensequenzen. Die Definition des $y$-Werts bleibt gleich.

Dimensionen des Trainingssatzes ($x$):
* Achse 1: Trainingssatzelemente (Sequenzen) (müssen die gleiche Größe wie die Größe $y$ haben)
* Achse 2: Mitglieder der Sequenz
* Achse 3: Merkmale in Daten (wie Eingangsneuronen)

Bisher haben wir als Eingabe einen einzelnen Aktienkurs verwendet, um vorherzusagen, ob wir kaufen (1), verkaufen (-1) oder halten (0) sollten. Der folgende Code veranschaulicht diese Kodierung.

In [2]:
x = [[32], [41], [39], [20], [15]]

y = [1, -1, 0, -1, 1]

print(x)
print(y)

[[32], [41], [39], [20], [15]]
[1, -1, 0, -1, 1]


Der folgende Code erstellt eine CSV-Datei von Grund auf. Um sie als Datenrahmen anzuzeigen, verwenden Sie Folgendes:

In [3]:
from IPython.display import display, HTML
import pandas as pd
import numpy as np

x = np.array(x)
print(x[:, 0])


df = DataFrame({"x": x[:, 0], "y": y})
display(df)

[32 41 39 20 15]


,x,y
0,32,1
1,41,-1
2,39,0
3,20,-1
4,15,1


Möglicherweise möchten Sie das Volumen mit dem Aktienkurs angeben. Der folgende Code zeigt, wie Sie eine Dimension zum Verwalten des Volumens hinzufügen.

In [4]:
x = [[32, 1383], [41, 2928], [39, 8823], [20, 1252], [15, 1532]]

y = [1, -1, 0, -1, 1]

print(x)
print(y)

[[32, 1383], [41, 2928], [39, 8823], [20, 1252], [15, 1532]]
[1, -1, 0, -1, 1]


Auch hier ist es sehr ähnlich zu dem, was wir zuvor gemacht haben. Im Folgenden wird dies als Datenrahmen angezeigt.

In [5]:
from IPython.display import display, HTML
import pandas as pd
import numpy as np

x = np.array(x)
print(x[:, 0])


df = DataFrame({"price": x[:, 0], "volume": x[:, 1], "y": y})
display(df)

[32 41 39 20 15]


,price,volume,y
0,32,1383,1
1,41,2928,-1
2,39,8823,0
3,20,1252,-1
4,15,1532,1


Nun kommen wir zum Sequenzformat. Wir wollen etwas über eine Sequenz vorhersagen, also muss das Datenformat eine Dimension hinzufügen. Sie müssen eine maximale Sequenzlänge angeben. Die einzelnen Sequenzen können beliebig groß sein.

In [6]:
x = [
    [[32, 1383], [41, 2928], [39, 8823], [20, 1252], [15, 1532]],
    [[35, 8272], [32, 1383], [41, 2928], [39, 8823], [20, 1252]],
    [[37, 2738], [35, 8272], [32, 1383], [41, 2928], [39, 8823]],
    [[34, 2845], [37, 2738], [35, 8272], [32, 1383], [41, 2928]],
    [[32, 2345], [34, 2845], [37, 2738], [35, 8272], [32, 1383]],
]

y = [1, -1, 0, -1, 1]

print(x)
print(y)

[[[32, 1383], [41, 2928], [39, 8823], [20, 1252], [15, 1532]], [[35, 8272], [32, 1383], [41, 2928], [39, 8823], [20, 1252]], [[37, 2738], [35, 8272], [32, 1383], [41, 2928], [39, 8823]], [[34, 2845], [37, 2738], [35, 8272], [32, 1383], [41, 2928]], [[32, 2345], [34, 2845], [37, 2738], [35, 8272], [32, 1383]]]
[1, -1, 0, -1, 1]


Auch wenn es nur ein Merkmal (Preis) gibt, müssen Sie 3 Dimensionen verwenden.

In [7]:
x = [
    [[32], [41], [39], [20], [15]],
    [[35], [32], [41], [39], [20]],
    [[37], [35], [32], [41], [39]],
    [[34], [37], [35], [32], [41]],
    [[32], [34], [37], [35], [32]],
]

y = [1, -1, 0, -1, 1]

print(x)
print(y)

[[[32], [41], [39], [20], [15]], [[35], [32], [41], [39], [20]], [[37], [35], [32], [41], [39]], [[34], [37], [35], [32], [41]], [[32], [34], [37], [35], [32]]]
[1, -1, 0, -1, 1]


# Modul 10 Aufgabe

Die erste Aufgabe findet ihr hier: [assignment 10](https://github.com/jeffheaton/app_deep_learning/blob/master/assignments/assignment_yourname_t81_558_class10.ipynb)